# RSNA Knee - v5 B2: train `effnet` (GPU T4 x2, Internet ON)

**This notebook:** EfficientNetV2-S, ImageNet-21k -> 1k. 0.923 for another team (LR 1.5e-4; 3e-4 diverged).

**Inputs to attach:**

1. the competition data
2. your `knee-cache-v5` dataset (from notebook A)
3. the label tables (at least *RSNA Knee LLM-read report labels*, pilkwang)

**Settings:** GPU **T4 x2**. **Internet ON** - only this training notebook needs it, to download the
ConvNeXt ImageNet weights once. The submission notebook (C) stays offline.

**Output:** `v5_fold*.pt` (the trained models), `config.json`, `oof_*.csv`.
After the run: **Output -> New Dataset -> `knee-v5-effnet`**.

What is new compared with v4 (each point comes from our research):

| Part | v4 | v5 |
|---|---|---|
| Backbone | ResNet34 | ConvNeXt-Tiny (ImageNet-22k -> 1k) |
| Pictures per knee | 12 groups of 3 | 42 sliding 3-slice pictures (train on 16 random ones, predict with all) |
| Head | mean + max | plane and position tags -> 2-layer transformer -> one attention per finding |
| Labels | pilkwang | average of every clean label table you attach (each one scored against gold first) |
| Folds | random | grouped by identical report text |
| Training | 5 epochs, LR 3e-4 for all | 12 epochs, backbone 1e-4, head 5e-4, 1-epoch warm-up then cosine, EMA of weights |

**Round 2 (v18):** attach the `knee-labels-round2` dataset (from notebook E) and this notebook trains on those
labels only, and saves the family as `<name>_r2`. Without it, nothing changes. At the end of every fold the
3 best rounds are averaged into one model (`SWA_TOP = 3`).


In [1]:
BACKBONE = "tf_efficientnetv2_s.in21k_ft_in1k"
FAMILY = "effnet"   # name for this model family in the submission

N_FOLDS = 5
FOLDS_TO_RUN = [0, 1, 2, 3, 4]                 # split over two sessions if needed, e.g. [0, 1, 2] then [3, 4]
EPOCHS = 12
BATCH_SIZE = 4                                 # knees per step
TRAIN_PICTURES = 16                            # random pictures per knee per step
LR_BACKBONE = 0.00015
LR_HEAD = 5e-4
WEIGHT_DECAY = 0.05
EMA_DECAY = 0.999
AUGMENT = True
GRAD_CHECKPOINT = "auto"                       # "auto": off; switched on by itself only if the GPU runs out of memory
ENC_SIZE = None                                # picture size the network reads (None = image store size, 288)
DROP_PATH = 0.0                                # random skipping of blocks while training (transformers: 0.1)
HEAD = "attn"                                  # "attn": one attention per finding; "query": DETR-style decoder with 12 finding queries
LAYER_DECAY = None                             # e.g. 0.8: lower layers of a transformer learn more slowly
LABEL_UNKNOWN = "keep"                        # "keep": use every answer (best models used this; "mask" lowered labels 0.900 -> 0.884)
                                               # "mask": drop a table's answer when its own confidence says
                                               # "not mentioned" (pilkwang: score 0.28, confidence 0.05);
                                               # "keep": use every value (how ResNet50 parts 0-1 were trained)
UNKNOWN_CONF = 0.5                             # confidence below this = "not mentioned"
MASK_EXACT_HALF = False                        # True: also treat an exact 0.5 as "not mentioned" (check the audit lines first)
MIRROR_CHECK = True                            # at the end, measure the mirror check on check and expert studies
ROUND2 = "auto"                                # "auto": if labels_round2.csv (notebook E) is attached, train on it only
SWA_TOP = 3                                    # at the end of a fold, average the weights of the 3 best rounds (1 = off)
LABEL_WEIGHTS = "auto"                         # "auto": confidence if only pilkwang, else 1
MIN_GOLD_AUC = 0.80                            # label tables weaker than this vs gold are not used
TIME_LIMIT_HOURS = 11.0                        # stop starting new folds after this
SEED = 42
PRETRAINED = True
NUM_WORKERS = 2                                # data loading processes (0 = load in the main process)
PRINT_EVERY = 50                               # a progress line every 50 steps
HEARTBEAT_SECONDS = 120                        # an "alive" line every 2 minutes, whatever is happening
STUCK_MINUTES = 10                             # no progress for this long -> print where the code is waiting
LOADER_TIMEOUT = 900                           # a data batch slower than this (seconds) stops with a clear error
MAX_BAD_STEPS = 20                             # skipped steps (GPU out of memory, bad loss) before stopping
ALLOW_CPU = False                              # True only for a quick test without a GPU
USE_BOTH_GPUS = True                           # False = one GPU only (slower, but the simplest possible setup)
MEM_STOP_GB = 28.0                             # programs' memory above this -> stop cleanly (Kaggle kills at about 30-32 GB)
assert set(FOLDS_TO_RUN) <= set(range(N_FOLDS)), f"FOLDS_TO_RUN must use numbers 0 to {N_FOLDS - 1}"

In [2]:
import sys
# Kaggle's newest image has a broken "triton" package, so "import timm" fails with
# "module 'triton' has no attribute 'backends'". PyTorch needs triton only for torch.compile,
# which we never use, so we always hide it (even if PyTorch is already loaded).
for _m in [m for m in sys.modules if m == "triton" or m.startswith("triton.")]:
    del sys.modules[_m]
sys.modules["triton"] = None
if "torch.utils._triton" in sys.modules:                 # forget an earlier "triton is there" answer
    for _f in ("has_triton_package", "has_triton"):
        _fn = getattr(sys.modules["torch.utils._triton"], _f, None)
        if hasattr(_fn, "cache_clear"):
            _fn.cache_clear()

import platform
try:
    import torch
    import timm
    import torchvision
except Exception as e:
    raise RuntimeError(f"PyTorch / timm still fail to load ({type(e).__name__}: {e}). "
                       "Click Run -> Restart & clear cell outputs, then run all cells again.") from e
print(f"python {platform.python_version()} | torch {torch.__version__} | timm {timm.__version__} | "
      f"torchvision {torchvision.__version__} | triton hidden")

import os
import glob
import json
import time
import random

import cv2
import numpy as np
import pandas as pd
import pydicom

try:
    from pydicom.pixels import apply_voi_lut
except ImportError:
    from pydicom.pixel_data_handlers.util import apply_voi_lut

COMP = next(p for p in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
                        "/kaggle/input/rsna-knee-abnormality-detection"] if os.path.exists(p))

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
PLANES = ["Sagittal", "Coronal", "Axial"]
print("data root:", COMP)

import copy
import psutil
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

import sys
import threading
import subprocess
import traceback

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")    # a stuck weight download fails instead of hanging
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")

RUN_START = time.time()
LOG_FILE = "/kaggle/working/train_log.txt"
STATE = {"stage": "starting", "last_progress": time.time()}


def log(msg):
    """Print with minutes since start, and keep a copy in train_log.txt."""
    line = f"[{(time.time() - RUN_START) / 60:7.1f} min] {msg}"
    print(line, flush=True)
    try:
        with open(LOG_FILE, "a") as f:
            f.write(line + "\n")
    except OSError:
        pass


def set_stage(stage):
    STATE["stage"] = stage
    STATE["last_progress"] = time.time()
    log(stage)


def ram_used_gb():
    """RAM as the Kaggle container counts it, else the whole machine."""
    for path in ("/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory/memory.usage_in_bytes"):
        try:
            return int(open(path).read()) / 1e9
        except (OSError, ValueError):
            pass
    vm = psutil.virtual_memory()
    return (vm.total - vm.available) / 1e9


import gc
import ctypes

try:
    _libc = ctypes.CDLL("libc.so.6")
except (OSError, AttributeError):
    _libc = None


def free_memory():
    """Give memory that Python and PyTorch no longer use back to the system."""
    gc.collect()
    if _libc is not None:
        _libc.malloc_trim(0)


def memory_parts():
    """Container memory in GB: programs (Python, PyTorch), shared (data workers), file cache (can be freed)."""
    stat = {}
    for path in ("/sys/fs/cgroup/memory.stat", "/sys/fs/cgroup/memory/memory.stat"):
        try:
            for line in open(path):
                k, v = line.split()
                stat[k] = int(v)
            break
        except (OSError, ValueError):
            continue
    programs = stat.get("anon", stat.get("total_rss", 0)) / 1e9
    shared = stat.get("shmem", stat.get("total_shmem", 0)) / 1e9
    cache = stat.get("file", stat.get("total_cache", 0)) / 1e9 - shared
    if not stat:                                   # no cgroup files (not on Kaggle): use this process
        programs = psutil.Process().memory_info().rss / 1e9
    return programs, shared, max(cache, 0.0)


def memory_text():
    programs, shared, cache = memory_parts()
    return (f"RAM {ram_used_gb():.1f} GB = programs {programs:.1f} + shared {shared:.1f} + file cache {cache:.1f} | "
            f"main process {psutil.Process().memory_info().rss / 1e9:.1f} GB")


def memory_too_high():
    programs, shared, _ = memory_parts()
    return programs + shared > MEM_STOP_GB


def gpu_status():
    """GPU use and memory from nvidia-smi ('' when there is no GPU)."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10).stdout
        parts = []
        for i, line in enumerate(out.strip().splitlines()):
            use, used, total = [v.strip() for v in line.split(",")]
            parts.append(f"GPU{i} {use}% busy, {int(used) / 1024:.1f}/{int(total) / 1024:.0f} GB")
        return " | ".join(parts)
    except Exception:
        return ""


def heartbeat():
    """Runs in the background for the whole session, so the log never goes silent."""
    main_id = threading.main_thread().ident
    while True:
        time.sleep(HEARTBEAT_SECONDS)
        quiet = (time.time() - STATE["last_progress"]) / 60
        log(f"alive | {STATE['stage']} | last progress {quiet:.0f} min ago | {memory_text()} | {gpu_status()}")
        if quiet >= STUCK_MINUTES:
            frame = sys._current_frames().get(main_id)
            if frame is not None:
                log("no progress for a long time - the code is waiting here:\n" + "".join(traceback.format_stack(frame)[-6:]))


if not any(t.name == "heartbeat" for t in threading.enumerate()):
    threading.Thread(target=heartbeat, name="heartbeat", daemon=True).start()
log("logging started: a progress line every PRINT_EVERY steps, an 'alive' line every HEARTBEAT_SECONDS")

python 3.13.15 | torch 2.11.0+cu128 | timm 1.0.29 | torchvision 0.26.0+cu128 | triton hidden
data root: /kaggle/input/competitions/rsna-knee-abnormality-detection
[    0.0 min] logging started: a progress line every PRINT_EVERY steps, an 'alive' line every HEARTBEAT_SECONDS


In [3]:
def plane_blocks(prep):
    """Where each plane's slices sit inside one study's array."""
    blocks, start = {}, 0
    for plane in PLANES:
        n = prep["slices"][plane]
        blocks[plane] = (start, start + n)
        start += n
    return blocks


def pick_series(series_df):
    """One series per plane. Fluid-sensitive first (in this data it always equals fat-suppressed)."""
    series_df = series_df.copy()
    for col, empty in (("Fluid_Sensitive", 0), ("Fat_Suppression", 0), ("Anatomical_Plane", ""), ("SeriesInstanceUID", "")):
        if col not in series_df.columns:
            series_df[col] = empty                    # a missing column must not crash the hidden test run
    if "StudyInstanceUID" not in series_df.columns or series_df.empty:
        return pd.DataFrame(columns=["StudyInstanceUID"] + PLANES)
    ranked = series_df.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
    rows = []
    for study, group in ranked.groupby("StudyInstanceUID"):
        row = {"StudyInstanceUID": str(study)}
        for plane in PLANES:
            same = group[group["Anatomical_Plane"] == plane]
            row[plane] = str(same.iloc[0]["SeriesInstanceUID"]) if len(same) else ""
        rows.append(row)
    return pd.DataFrame(rows)


def slice_key(header):
    """Position along the body axis the stack runs along (x for sagittal, y for coronal, z for axial).
    Using the body axis, not the scanner's slice direction, gives the same order on every scanner."""
    try:
        pos = np.array(header.ImagePositionPatient, dtype=float)
        orient = np.array(header.ImageOrientationPatient, dtype=float)
        normal = np.cross(orient[:3], orient[3:])
        return float(pos[int(np.argmax(np.abs(normal)))])
    except Exception:
        return float(getattr(header, "InstanceNumber", 0) or 0)


def sorted_paths(folder):
    ordered = []
    for path in glob.glob(os.path.join(folder, "*.dcm")):
        try:
            header = pydicom.dcmread(path, stop_before_pixels=True,
                                     specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"])
            ordered.append((slice_key(header), path))
        except Exception:
            continue
    ordered.sort(key=lambda item: item[0])
    return [path for _, path in ordered]


def read_image(path):
    """One slice as floats, plus mm per pixel along rows and columns."""
    try:
        data = pydicom.dcmread(path)
        try:
            image = apply_voi_lut(data.pixel_array, data).astype(np.float32)
        except Exception:
            image = np.asarray(data.pixel_array, dtype=np.float32)
        image = np.nan_to_num(np.squeeze(image), nan=0.0, posinf=0.0, neginf=0.0)
        if image.ndim != 2:
            return None, None
        if getattr(data, "PhotometricInterpretation", "") == "MONOCHROME1":
            image = image.max() - image
        spacing = getattr(data, "PixelSpacing", None)
        mm = (float(spacing[0]), float(spacing[1])) if spacing is not None and len(spacing) >= 2 else None
        return image, mm
    except Exception:
        return None, None


def crop_to_mm(image, mm, crop_mm):
    """Keep the same real area of knee, whatever the scanner's pixel size."""
    if not mm or mm[0] <= 0 or mm[1] <= 0:
        return image
    half_y, half_x = int(round(crop_mm / mm[0] / 2)), int(round(crop_mm / mm[1] / 2))
    mid_y, mid_x = image.shape[0] // 2, image.shape[1] // 2
    cropped = image[max(mid_y - half_y, 0):mid_y + half_y, max(mid_x - half_x, 0):mid_x + half_x]
    return cropped if cropped.size else image


def load_series(folder, n, prep):
    size = prep["img_size"]
    empty = np.zeros((n, size, size), dtype=np.uint8)
    paths = sorted_paths(folder)
    if not paths:
        return empty

    start = int(prep["span"][0] * len(paths))
    stop = max(int(prep["span"][1] * len(paths)), start + 1)
    picked = np.linspace(start, stop - 1, n).round().astype(int)

    slices, last_good = [], None
    for i in picked:
        image, mm = read_image(paths[i])
        if image is None:
            slices.append(last_good if last_good is not None else np.zeros((size, size), np.float32))
            continue
        image = cv2.resize(crop_to_mm(image, mm, prep["crop_mm"]), (size, size), interpolation=cv2.INTER_AREA)
        slices.append(image)
        last_good = image

    stack = np.stack(slices).astype(np.float32)
    low, high = np.percentile(stack, [1, 99])         # one window for the whole series
    stack = (np.clip(stack, low, high) - low) / max(high - low, 1e-6)
    return (stack * 255).astype(np.uint8)


def knee_side(study_uid, row, root):
    """'L', 'R' or None. Laterality tag first, else where the image centre sits (right knee at x < 0)."""
    tags, centres = [], []
    for plane in PLANES:
        series_uid = row.get(plane, "")
        if not isinstance(series_uid, str) or not series_uid:
            continue
        files = glob.glob(os.path.join(root, study_uid, series_uid, "*.dcm"))
        if not files:
            continue
        try:
            h = pydicom.dcmread(files[0], stop_before_pixels=True)
        except Exception:
            continue
        tag = str(getattr(h, "Laterality", "") or "").strip().upper()
        if tag[:1] in ("L", "R"):
            tags.append(tag[0])
        try:
            pos = np.array(h.ImagePositionPatient, dtype=float)
            orient = np.array(h.ImageOrientationPatient, dtype=float)
            spacing = np.array(h.PixelSpacing, dtype=float)
            centre = (pos + orient[:3] * spacing[1] * float(h.Columns) / 2
                          + orient[3:] * spacing[0] * float(h.Rows) / 2)
            centres.append(centre[0])
        except Exception:
            pass
    if tags:
        return tags[0]
    if centres and abs(float(np.median(centres))) >= 20:
        return "R" if float(np.median(centres)) < 0 else "L"
    return None


def load_study(study_uid, row, root, prep):
    """One study as (all slices, size, size) uint8: sagittal block, coronal block, axial block.
    Left knees are mirrored so every knee looks like a right knee. Missing planes stay black."""
    blocks = plane_blocks(prep)
    total = blocks[PLANES[-1]][1]
    volume = np.zeros((total, prep["img_size"], prep["img_size"]), dtype=np.uint8)

    for plane in PLANES:
        series_uid = row.get(plane, "")
        if isinstance(series_uid, str) and series_uid:
            a, b = blocks[plane]
            try:
                volume[a:b] = load_series(os.path.join(root, study_uid, series_uid), b - a, prep)
            except Exception:
                pass

    try:
        if knee_side(study_uid, row, root) == "L":
            a, b = blocks["Sagittal"]
            volume[a:b] = volume[a:b][::-1]                  # sagittal: read the stack backwards
            for plane in ("Coronal", "Axial"):
                a, b = blocks[plane]
                volume[a:b] = volume[a:b, :, ::-1]           # coronal / axial: flip left-right
    except Exception:
        pass
    return volume

In [4]:
import torch
import torch.nn as nn
import timm
import torchvision
from concurrent.futures import ThreadPoolExecutor, wait

# RadImageNet: CNNs pretrained on 1.35 million radiology images (CT, MRI, ultrasound), MIT licence
RAD_FILES = {"radimagenet_resnet50": "ResNet50.pt",
             "radimagenet_densenet121": "DenseNet121.pt",
             "radimagenet_inceptionv3": "InceptionV3.pt"}


class RadTrunk(nn.Module):
    """torchvision ResNet50 / DenseNet121 / InceptionV3 without the classifier."""

    def __init__(self, name):
        super().__init__()
        self.name = name
        if name == "radimagenet_resnet50":
            self.net = nn.Sequential(*list(torchvision.models.resnet50(weights=None).children())[:-2])
            self.num_features = 2048
        elif name == "radimagenet_densenet121":
            self.net = torchvision.models.densenet121(weights=None, memory_efficient=False).features
            self.num_features = 1024
        else:
            m = torchvision.models.inception_v3(weights=None, aux_logits=False, init_weights=False)
            m.fc, m.dropout = nn.Identity(), nn.Identity()
            self.net = m
            self.num_features = 2048

    def forward(self, x):
        y = self.net(x)
        if self.name == "radimagenet_densenet121":
            y = torch.relu(y)
        return y.mean(dim=(2, 3)) if y.ndim == 4 else y

    def set_grad_checkpointing(self, enable=True):
        for m in self.net.modules():            # DenseNet layers have their own memory-saving switch
            if hasattr(m, "memory_efficient"):
                m.memory_efficient = enable


def find_rad_file(name):
    hits = sorted(glob.glob(f"/kaggle/input/datasets/*/*/{name}") + glob.glob(f"/kaggle/input/datasets/*/*/*/{name}"))
    if hits:
        return hits[0]
    from huggingface_hub import hf_hub_download        # needs Internet ON (training notebooks only)
    return hf_hub_download("Lab-Rasool/RadImageNet", name)


def load_rad_weights(trunk, path):
    """The files come in different key layouts, so tensors are matched in order and every shape is checked."""
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, nn.Module):
        obj = obj.state_dict()
    if isinstance(obj, dict) and "state_dict" in obj:
        obj = obj["state_dict"]
    skip = ("fc.", "classifier.", "AuxLogits.", "num_batches_tracked")
    src = [v for k, v in obj.items() if torch.is_tensor(v) and not any(t in k for t in skip)]
    dst = [k for k in trunk.net.state_dict() if not any(t in k for t in skip)]
    shapes_ok = len(src) == len(dst) and all(tuple(v.shape) == tuple(trunk.net.state_dict()[k].shape) for v, k in zip(src, dst))
    if not shapes_ok:
        raise ValueError(f"RadImageNet file {path} does not match {trunk.name}: {len(src)} tensors vs {len(dst)} expected")
    trunk.net.load_state_dict(dict(zip(dst, src)), strict=False)
    print(f"RadImageNet weights loaded: {path} ({len(dst)} tensors, all shapes match)")


def build_encoder(backbone, pretrained=False, enc_size=None, drop_path=0.0):
    """Returns the image network and the input scaling its weights expect."""
    if backbone in RAD_FILES:
        trunk = RadTrunk(backbone)
        if pretrained:
            load_rad_weights(trunk, find_rad_file(RAD_FILES[backbone]))
        return trunk, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)      # RadImageNet expects -1..1
    kw = {"num_classes": 0}
    if drop_path:
        kw["drop_path_rate"] = drop_path
    try:
        # transformers and CoAtNet are built for one picture size; timm resizes their position tables
        enc = timm.create_model(backbone, pretrained=pretrained, img_size=enc_size, **kw) if enc_size else \
            timm.create_model(backbone, pretrained=pretrained, **kw)
    except TypeError:                                     # plain CNNs take any size and have no img_size
        enc = timm.create_model(backbone, pretrained=pretrained, **kw)
    cfg = enc.pretrained_cfg
    return enc, cfg.get("mean", (0.485, 0.456, 0.406)), cfg.get("std", (0.229, 0.224, 0.225))


def window_table(prep):
    """Every sliding 3-slice picture: (plane id, centre slice, position in the plane)."""
    rows = []
    for p, plane in enumerate(PLANES):
        a, b = plane_blocks(prep)[plane]
        for c in range(a + 1, b - 1):
            rows.append((p, c, c - a))
    return np.array(rows, dtype=np.int64)


def make_windows(volume, chosen, table):
    """(len(chosen), 3, H, W) floats in 0-1, plus plane ids, positions and a valid flag per picture."""
    rows = table[chosen]
    centres = rows[:, 1]
    x = np.stack([volume[c - 1:c + 2] for c in centres]).astype(np.float32) / 255.0
    valid = x.reshape(len(rows), -1).max(axis=1) > 0
    return x, rows[:, 0], rows[:, 2], valid


class KneeNetV5(nn.Module):
    """Backbone per picture -> plane and position tags -> small transformer -> one attention per finding."""

    def __init__(self, backbone, pretrained=False, dim=256, max_pos=64, enc_size=None, drop_path=0.0, head="attn"):
        super().__init__()
        self.enc_size, self.head = enc_size, head
        self.encoder, mean, std = build_encoder(backbone, pretrained, enc_size, drop_path)
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

        self.proj = nn.Linear(self.encoder.num_features, dim)
        self.plane_emb = nn.Embedding(len(PLANES), dim)
        self.pos_emb = nn.Embedding(max_pos, dim)
        layer = nn.TransformerEncoderLayer(dim, 4, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
        self.mixer = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        if head == "query":
            # DETR-style: one learnable query per finding; each query reads the pictures through cross-attention
            self.queries = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)
            dec = nn.TransformerDecoderLayer(dim, 4, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
            self.decoder = nn.TransformerDecoder(dec, 2)
            self.dec_norm = nn.LayerNorm(dim)
        else:
            self.attn = nn.Linear(dim, len(LABELS))
        self.out_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)
        self.out_b = nn.Parameter(torch.zeros(len(LABELS)))

    def forward(self, x, plane, pos, valid):            # x: (batch, pictures, 3, H, W)
        b, w = x.shape[:2]
        x = x.flatten(0, 1)
        if self.enc_size and x.shape[-1] != self.enc_size:      # e.g. 288 -> 280 for 14-pixel patches
            x = nn.functional.interpolate(x, size=(self.enc_size, self.enc_size), mode="bilinear", align_corners=False)
        feats = self.encoder((x - self.mean) / self.std).reshape(b, w, -1)
        h = self.proj(feats) + self.plane_emb(plane) + self.pos_emb(pos)

        valid = valid.clone()
        valid[valid.sum(1) == 0] = True                  # a study with no pictures: avoid empty attention
        h = self.norm(self.mixer(h, src_key_padding_mask=~valid))

        if self.head == "query":
            q = self.queries.unsqueeze(0).expand(b, -1, -1).to(h.dtype)
            pooled = self.dec_norm(self.decoder(q, h, memory_key_padding_mask=~valid)).float()   # one summary per finding
        else:
            scores = self.attn(h).float().masked_fill(~valid[..., None], -1e4).softmax(dim=1)
            pooled = torch.einsum("bwk,bwd->bkd", scores, h.float())    # one summary per finding
        return (pooled * self.out_w).sum(-1) + self.out_b


class SteadyDataParallel(nn.DataParallel):
    """Same as DataParallel, but the second GPU always runs in the same helper thread.
    Normal DataParallel starts new threads at every step. PyTorch keeps its cuDNN plan cache
    per thread and never frees it, so every step rebuilt every plan on the CPU (GPU idle)
    and the old plans stayed in memory (RAM grew every step)."""

    _helpers = None

    def parallel_apply(self, replicas, inputs, kwargs):
        if SteadyDataParallel._helpers is None:
            SteadyDataParallel._helpers = ThreadPoolExecutor(max_workers=max(1, len(self.device_ids) - 1))
        grad, amp = torch.is_grad_enabled(), torch.is_autocast_enabled("cuda")

        def run(i):
            torch.set_grad_enabled(grad)
            with torch.cuda.device(self.device_ids[i]), torch.amp.autocast("cuda", enabled=amp):
                return replicas[i](*inputs[i], **kwargs[i])

        later = [SteadyDataParallel._helpers.submit(run, i) for i in range(1, len(replicas))]
        try:
            first = run(0)                       # GPU 0 works in the main thread
        finally:
            wait(later)                          # never leave the helper half-way through a step
        return [first] + [f.result() for f in later]


# Mirror check at test time: a mirrored right knee looks like a left knee. The model only learned right knees,
# so on the mirrored copy its medial and lateral answers come out swapped; we swap them back.
# MCL has no lateral partner among the 12 findings, so MCL always comes from the normal copy only.
SWAP = [LABELS.index(n) for n in ["ACL", "MCL", "Lateral Meniscus", "Medial Meniscus",
                                  "Lateral OA", "Medial OA", "PF OA", "Effusion",
                                  "Synovitis", "Baker's", "Contusion", "Fracture"]]
MCL_I = LABELS.index("MCL")


def mirror_volume(volume, prep):
    """The same knee, mirrored: side-view stack read backwards, front and top views flipped left-right."""
    v = volume.copy()
    blocks = plane_blocks(prep)
    a, b = blocks["Sagittal"]
    v[a:b] = v[a:b][::-1]
    for plane in ("Coronal", "Axial"):
        a, b = blocks[plane]
        v[a:b] = v[a:b, :, ::-1]
    return np.ascontiguousarray(v)


def join_mirror(logits, logits_mirror):
    """Average the normal answers with the swapped-back mirrored answers (MCL: normal only)."""
    out = (logits + logits_mirror[:, SWAP]) / 2
    out[:, MCL_I] = logits[:, MCL_I]
    return out

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPUS = torch.cuda.device_count()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE, " GPUs:", N_GPUS, " timm:", timm.__version__)
print(f"RAM: {psutil.virtual_memory().available / 1e9:.1f} GB free")

if DEVICE != "cuda" and not ALLOW_CPU:
    raise RuntimeError("No GPU found. Settings -> Accelerator -> GPU T4 x2, then run again.")
for i in range(N_GPUS):
    log(f"GPU {i}: {torch.cuda.get_device_name(i)}")
log(f"GPU status: {gpu_status() or 'no GPU'}")

device: cuda  GPUs: 2  timm: 1.0.29
RAM: 31.7 GB free
[    0.0 min] GPU 0: Tesla T4
[    0.0 min] GPU 1: Tesla T4
[    0.0 min] GPU status: GPU0 0% busy, 0.0/15 GB | GPU1 0% busy, 0.0/15 GB


## 1. The image cache from notebook A

In [5]:
set_stage("opening the image store")
# any attached folder with prep.json + cache.npy + uids.csv is an image store, whatever the dataset is called
caches = sorted({os.path.dirname(p) for depth in range(1, 5) for p in glob.glob("/kaggle/input/" + "*/" * depth + "prep.json")
                 if os.path.isfile(os.path.join(os.path.dirname(p), "cache.npy"))
                 and os.path.isfile(os.path.join(os.path.dirname(p), "uids.csv"))})
assert caches, "Attach your image store dataset (output of the Image Preparation notebook: cache.npy, uids.csv, prep.json)."
if len(caches) > 1:
    log(f"more than one image store attached, using the first: {caches}")
CACHE_DIR = caches[0]
prep = json.load(open(f"{CACHE_DIR}/prep.json"))
cache_uids = pd.read_csv(f"{CACHE_DIR}/uids.csv")["StudyInstanceUID"].astype(str).tolist()
row_of = {u: i for i, u in enumerate(cache_uids)}


class CacheReader:
    """Reads one study at a time from cache.npy, then drops it from RAM.
    A memory map (mmap_mode="r") kept every study it had read in RAM, in the main process and in
    each data worker, so a 17.5 GB cache crashed the notebook. Here RAM stays flat."""

    def __init__(self, path):
        header = np.load(path, mmap_mode="r")           # only to get the shape and where the data starts
        self.shape, self.dtype, self.offset = header.shape, header.dtype, header.offset
        del header
        self.study_bytes = int(np.prod(self.shape[1:])) * self.dtype.itemsize
        self.nbytes = self.shape[0] * self.study_bytes
        self.fd = os.open(path, os.O_RDONLY)

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, i):
        start = self.offset + int(i) * self.study_bytes
        data = os.pread(self.fd, self.study_bytes, start)
        if len(data) != self.study_bytes:
            raise IOError(f"cache.npy: short read for row {i}")
        try:
            os.posix_fadvise(self.fd, start, self.study_bytes, os.POSIX_FADV_DONTNEED)   # forget it after reading
        except (AttributeError, OSError):
            pass
        return np.frombuffer(data, dtype=self.dtype).reshape(self.shape[1:])


volumes = CacheReader(f"{CACHE_DIR}/cache.npy")
assert len(volumes) == len(cache_uids), f"cache.npy has {len(volumes)} rows but uids.csv has {len(cache_uids)}"
t0 = time.time()
_ = volumes[len(volumes) - 1]                          # read the last study once: proves the file is complete
log(f"cache: {CACHE_DIR} {volumes.shape}, {volumes.nbytes / 1e9:.1f} GB on disk; one study read in {time.time() - t0:.2f} s")
log(f"RAM used: {ram_used_gb():.1f} GB, free: {psutil.virtual_memory().available / 1e9:.1f} GB")
log(f"image settings: {prep}")

TABLE = window_table(prep)
log(f"pictures per knee: {len(TABLE)}")

[    0.0 min] opening the image store
[    0.2 min] cache: /kaggle/input/datasets/tanishakeshav/knee-cache-vishupal (4407, 48, 288, 288), 17.5 GB on disk; one study read in 0.03 s
[    0.2 min] RAM used: 1.5 GB, free: 31.8 GB
[    0.2 min] image settings: {'slices': {'Sagittal': 20, 'Coronal': 16, 'Axial': 12}, 'img_size': 288, 'crop_mm': 140.0, 'span': [0.06, 0.94], 'version': 'v5-body-axis-right-knees'}
[    0.2 min] pictures per knee: 42


## 2. Labels: every clean table, scored against the 58 expert studies

The notebook looks at every CSV in the attached datasets that has a `StudyInstanceUID` column and the
12 findings. Each table is scored against the 58 gold studies:

- **0.98 or more = it copies the gold answers.** It is not used (some public tables do this).
- **below 0.80 = too weak.** Not used.

The rest are averaged. Gold studies are never trained on, whatever the table says.

In [6]:
train = pd.read_csv(f"{COMP}/train.csv")
train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
gold = train.dropna(subset=LABELS).reset_index(drop=True)
missing_gold = (~gold["StudyInstanceUID"].isin(row_of)).sum()
if missing_gold:
    log(f"warning: {missing_gold} expert studies are not in the image store; they are left out of the expert score")
    gold = gold[gold["StudyInstanceUID"].isin(row_of)].reset_index(drop=True)
gold_ids = set(gold["StudyInstanceUID"])
print("studies:", len(train), " gold:", len(gold))


def norm(name):
    return "".join(ch for ch in str(name).lower() if ch.isalnum())


ALIASES = {norm(l): l for l in LABELS}
ALIASES.update({"bakers": "Baker's", "bakerscyst": "Baker's", "pfoa": "PF OA", "patellofemoraloa": "PF OA"})


def read_table(path):
    """A label table as (study, 12 findings) in 0-1, plus confidence columns if it has them."""
    try:
        t = pd.read_csv(path)
    except Exception:
        return None, None
    if "StudyInstanceUID" not in t.columns:
        return None, None
    found, conf = {}, {}
    for col in t.columns:
        key = norm(col)
        for suffix in ("conf", "confidence"):
            if key.endswith(suffix) and key[:-len(suffix)] in ALIASES:
                conf[ALIASES[key[:-len(suffix)]]] = col
        for suffix in ("", "prob", "score", "p", "pred", "soft"):
            base = key[:-len(suffix)] if suffix and key.endswith(suffix) else key
            if base in ALIASES and ALIASES[base] not in found and "conf" not in key:
                found[ALIASES[base]] = col
    if len(found) < 12:
        return None, None
    out = pd.DataFrame({"StudyInstanceUID": t["StudyInstanceUID"].astype(str)})
    for label in LABELS:
        values = pd.to_numeric(t[found[label]], errors="coerce")
        if values.isna().all():
            return None, None
        top = values.max()
        if top > 10:
            values = values / 100            # written as 0-100
        elif top > 1.5:
            values = values / top            # a grade scale such as 0-3
        out[label] = values.clip(0, 1)
    out = out.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")
    weights = None
    if len(conf) == 12:
        weights = pd.DataFrame({l: pd.to_numeric(t[conf[l]], errors="coerce").values for l in LABELS},
                               index=t["StudyInstanceUID"].astype(str))
        weights = weights[~weights.index.duplicated()]
    return out, weights


def gold_auc(table):
    rows = gold.set_index("StudyInstanceUID").join(table, rsuffix="_t", how="inner")
    scores = []
    for label in LABELS:
        y, p = rows[label].values, rows[label + "_t"].values
        ok = ~np.isnan(p)
        if ok.sum() > 5 and len(np.unique(y[ok])) == 2:
            scores.append(roc_auc_score(y[ok], p[ok]))
    return (float(np.mean(scores)) if scores else None), len(rows)


tables, confs = [], []
skip = ("train.csv", "test.csv", "train_series.csv", "test_series.csv", "sample_submission.csv", "uids.csv", "submission.csv")
csv_paths = sorted({q for depth in range(1, 5) for q in glob.glob("/kaggle/input/" + "*/" * depth + "*.csv")})
round2_paths = [p for p in csv_paths if os.path.basename(p) == "labels_round2.csv"]
USE_ROUND2 = bool(round2_paths) and ROUND2 in ("auto", True)
if USE_ROUND2:
    # round 2: one table from notebook E = report labels mixed with our first models' out-of-fold answers
    r2 = pd.read_csv(round2_paths[0])
    r2["StudyInstanceUID"] = r2["StudyInstanceUID"].astype(str)
    r2 = r2.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")
    tables = [r2[LABELS].astype("float32")]
    confs = [r2[["w_" + l for l in LABELS]].set_axis(LABELS, axis=1).astype("float32")]
    if len(round2_paths) > 1:
        log(f"warning: more than one labels_round2.csv attached, using the first: {round2_paths}")
    info_path = os.path.join(os.path.dirname(round2_paths[0]), "round2_info.json")
    round_tag = json.load(open(info_path)).get("round_name", "r2") if os.path.isfile(info_path) else "r2"
    if not FAMILY.endswith("_" + round_tag):
        FAMILY = FAMILY + "_" + round_tag
    log(f"round-2 labels: {round2_paths[0]} ({len(r2)} studies); the other label tables are not read. "
        f"This model family is saved as '{FAMILY}'")
    csv_paths = []
for path in csv_paths:
    folder, name = os.path.dirname(path), os.path.basename(path)
    if (name in skip or path.startswith(COMP) or path.startswith(CACHE_DIR)
            or name.startswith(("oof_fold", "gold_fold")) or os.path.isfile(os.path.join(folder, "config.json"))):
        continue                          # competition files, the image store, and our own model outputs are not labels
    table, weights = read_table(path)
    if table is None:
        continue
    auc, n_gold = gold_auc(table)
    status = "use"
    if auc is None:
        status = "SKIP: no expert studies to check it"
    elif auc >= 0.98:
        status = "SKIP: copies the gold answers"
    elif auc < MIN_GOLD_AUC:
        status = "SKIP: too weak"
    print(f"{status:32s} gold AUC {auc if auc is None else round(auc, 3)}  rows {len(table)}  {path}")
    if status == "use":
        tables.append(table)
        confs.append(weights)

assert tables, "No label table found. Attach at least 'RSNA Knee LLM-read report labels' (pilkwang)."

studies: 4407  gold: 58
[    0.5 min] round-2 labels: /kaggle/input/datasets/vishupalgoyal23/knee-labels-round2/labels_round2.csv (4349 studies); the other label tables are not read. This model family is saved as 'effnet_r2'


In [7]:
# Audit: how each table writes "not mentioned". A missing answer must not count as a real one.
for k, table in enumerate(tables):
    c = confs[k].reindex(table.index) if confs[k] is not None else None
    low = (c < UNKNOWN_CONF).mean() if c is not None else None
    half = (table == 0.5).mean()
    print(f"table {k + 1}: confidence columns {'yes' if c is not None else 'no'}")
    print("  " + "  ".join(f"{l[:9]}: empty {table[l].isna().mean():.0%} half {half[l]:.0%}"
                           + (f" low-conf {low[l]:.0%}" if low is not None else "") for l in LABELS))
    if LABEL_UNKNOWN == "mask" and not USE_ROUND2:      # round-2 labels already carry their own weights
        if c is not None:
            tables[k] = table.where(c >= UNKNOWN_CONF)             # low confidence -> no answer from this table
        if MASK_EXACT_HALF:
            tables[k] = tables[k].mask(tables[k] == 0.5)
conf_tables = [c for c in confs if c is not None]
print(f"label mode: {LABEL_UNKNOWN}")

targets = pd.concat(tables).groupby(level=0).mean()          # average of the clean tables, cell by cell
labels = train[["StudyInstanceUID"]].join(targets, on="StudyInstanceUID")

if LABEL_WEIGHTS == "auto" and len(tables) == 1 and conf_tables:
    w = train[["StudyInstanceUID"]].join(conf_tables[0], on="StudyInstanceUID")[LABELS].fillna(0.0)
    print("one table with confidences: using them as weights")
else:
    w = labels[LABELS].notna().astype("float32")
    print(f"{len(tables)} tables averaged: weight 1 where a label exists")

labels[LABELS] = labels[LABELS].fillna(0.5)
avg_auc, _ = gold_auc(targets)
print("averaged labels vs gold:", "n/a" if avg_auc is None else f"{avg_auc:.3f}")

# training studies: in the cache, not gold, with at least one label
usable = [i for i, u in enumerate(train["StudyInstanceUID"]) if u not in gold_ids and u in row_of and w.iloc[i].sum() > 0]
uids_tr = train["StudyInstanceUID"].values[usable]
rows_tr = np.array([row_of[u] for u in uids_tr])
y = labels[LABELS].values[usable].astype("float32")
weight = w[LABELS].values[usable].astype("float32")
rows_gold = np.array([row_of[u] for u in gold["StudyInstanceUID"]])
y_gold = gold[LABELS].values.astype("float32")
print("training studies:", len(rows_tr))
print("share of training studies with an answer, per finding:")
print("  " + "  ".join(f"{l[:9]} {weight[:, i].mean():.0%}" for i, l in enumerate(LABELS)))

table 1: confidence columns yes
  ACL: empty 0% half 0% low-conf 0%  MCL: empty 0% half 0% low-conf 0%  Medial Me: empty 0% half 0% low-conf 0%  Lateral M: empty 0% half 0% low-conf 0%  Medial OA: empty 0% half 0% low-conf 0%  Lateral O: empty 0% half 0% low-conf 0%  PF OA: empty 0% half 0% low-conf 0%  Effusion: empty 0% half 0% low-conf 0%  Synovitis: empty 0% half 0% low-conf 0%  Baker's: empty 0% half 0% low-conf 0%  Contusion: empty 0% half 0% low-conf 0%  Fracture: empty 0% half 0% low-conf 0%
label mode: keep
one table with confidences: using them as weights
averaged labels vs gold: n/a
training studies: 4349
share of training studies with an answer, per finding:
  ACL 100%  MCL 100%  Medial Me 100%  Lateral M 100%  Medial OA 100%  Lateral O 100%  PF OA 100%  Effusion 100%  Synovitis 100%  Baker's 100%  Contusion 100%  Fracture 100%


## 3. Folds grouped by report text

Many reports are word-for-word the same ("normal knee" templates). Grouping by report text keeps those
studies in one fold, so the check set is honest. One team found random folds looked ~0.05 better than
they really were.

In [8]:
from sklearn.model_selection import GroupKFold
import hashlib

if "Report" in train.columns:
    report_group = train["Report"].fillna("").str.lower().str.replace(r"\s+", " ", regex=True).values[usable]
else:
    report_group = train["StudyInstanceUID"].values[usable]
fold_of = np.zeros(len(rows_tr), dtype=int)
order = np.random.default_rng(SEED).permutation(len(rows_tr))       # shuffle so GroupKFold is not by row order
for f, (_, valid_part) in enumerate(GroupKFold(N_FOLDS).split(order, groups=report_group[order])):
    fold_of[order[valid_part]] = f
assert not set(uids_tr) & gold_ids
for f in range(N_FOLDS):
    print(f"fold {f}: train {(fold_of != f).sum()}   check {(fold_of == f).sum()}")

# The folds depend on which label tables are attached. Attach the same tables in every session;
# this fingerprint must then be the same each time (the prediction notebook checks it).
FOLDS_ID = hashlib.md5(",".join(f"{u}:{f}" for u, f in sorted(zip(uids_tr, fold_of))).encode()).hexdigest()[:8]
log(f"fold fingerprint: {FOLDS_ID}")

fold 0: train 3479   check 870
fold 1: train 3479   check 870
fold 2: train 3479   check 870
fold 3: train 3479   check 870
fold 4: train 3480   check 869
[    0.6 min] fold fingerprint: 5f145e75


## 4. Data and model

In [9]:
set_stage("preparing data loaders")


def macro_auc(y_true, y_pred, mask=None):
    """Average AUC over the problems that have both classes. NaN if none has."""
    scores = {}
    for i, name in enumerate(LABELS):
        t, p = y_true[:, i], y_pred[:, i]
        if mask is not None:
            keep = mask[:, i] > 0.5
            t, p = t[keep], p[keep]
        ok = np.isfinite(p)
        t, p = (t[ok] > 0.5).astype(int), p[ok]
        if len(np.unique(t)) == 2:
            scores[name] = roc_auc_score(t, p)
    scores["MACRO"] = float(np.mean(list(scores.values()))) if scores else float("nan")
    return scores


def show(scores):
    for name in LABELS + ["MACRO"]:
        if name in scores:
            print(f"  {name:18s} {scores[name]:.3f}")


def augment_plane(stack):
    """Same small rotation, zoom and shift for every slice of one plane. Never a flip."""
    n, h, w = stack.shape
    m = cv2.getRotationMatrix2D((w / 2, h / 2), np.random.uniform(-10, 10), np.random.uniform(0.9, 1.1))
    m[:, 2] += np.random.uniform(-0.05, 0.05, size=2) * (w, h)
    return np.stack([cv2.warpAffine(s, m, (w, h), flags=cv2.INTER_LINEAR, borderValue=0) for s in stack])


class KneeSet(Dataset):
    def __init__(self, rows, y, weight, training, mirror=False):
        self.rows, self.y, self.weight, self.training, self.mirror = rows, y, weight, training, mirror

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, k):
        volume = volumes[self.rows[k]]
        if self.mirror:
            volume = mirror_volume(volume, prep)
        if self.training:
            if AUGMENT:
                volume = volume.copy()
                for plane in PLANES:
                    a, b = plane_blocks(prep)[plane]
                    if np.random.rand() < 0.7:
                        volume[a:b] = augment_plane(volume[a:b])
            # random pictures, preferring planes that exist
            present = np.array([volume[c].max() > 0 for c in TABLE[:, 1]])
            ids = np.random.permutation(np.where(present)[0]).tolist() + np.random.permutation(np.where(~present)[0]).tolist()
            chosen = np.sort(np.array(ids[:TRAIN_PICTURES]))
        else:
            chosen = np.arange(len(TABLE))
        x, plane, pos, valid = make_windows(volume, chosen, TABLE)
        if self.training and AUGMENT:
            x = np.clip(x ** np.random.uniform(0.85, 1.15) * np.random.uniform(0.9, 1.1), 0, 1)
        return (torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)), torch.from_numpy(plane), torch.from_numpy(pos),
                torch.from_numpy(valid), torch.from_numpy(self.y[k]), torch.from_numpy(self.weight[k]))


def worker_init(worker_id):
    # forked workers would all share one numpy random state, so give each its own;
    # OpenCV's own threads can freeze inside forked workers, so switch them off
    np.random.seed(torch.initial_seed() % 2**32)
    random.seed(torch.initial_seed())
    cv2.setNumThreads(0)
    torch.set_num_threads(1)


def loader(rows, y, weight, training, mirror=False):
    return DataLoader(KneeSet(rows, y, weight, training, mirror), batch_size=BATCH_SIZE, shuffle=training,
                      drop_last=training, num_workers=NUM_WORKERS, pin_memory=False,   # pinned memory grew every round and was never given back
                      worker_init_fn=worker_init if NUM_WORKERS > 0 else None,
                      timeout=LOADER_TIMEOUT if NUM_WORKERS > 0 else 0, persistent_workers=False,
                      multiprocessing_context="fork" if NUM_WORKERS > 0 else None)


def predict(model, dl, name):
    model.eval()
    out, t0 = [], time.time()
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
        for i, (x, plane, pos, valid, _, _) in enumerate(dl):
            logits = model(x.to(DEVICE), plane.to(DEVICE), pos.to(DEVICE), valid.to(DEVICE))
            out.append(logits.float().cpu().numpy())
            STATE["last_progress"] = time.time()
            if (i + 1) % PRINT_EVERY == 0 or i + 1 == len(dl):
                log(f"  scoring {name}: batch {i + 1}/{len(dl)}  {(time.time() - t0) / (i + 1):.2f} s per batch")
    return np.concatenate(out)


def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -50, 50)))


cv2.setNumThreads(0)
gold_dl = loader(rows_gold, y_gold, np.ones_like(y_gold), False)
log(f"data loaders ready: {NUM_WORKERS} workers, batch {BATCH_SIZE} knees, {TRAIN_PICTURES} pictures per knee while training")

[    0.6 min] preparing data loaders
[    0.6 min] data loaders ready: 2 workers, batch 4 knees, 16 pictures per knee while training


## 5. Train

For every fold: train 12 epochs, keep an **EMA** (a slowly moving average of the weights, which is
steadier than the last step), score it on the fold's check set after every epoch, and save the best epoch.
Gold is printed but never used to choose.

In [10]:
def save_fold(fold, state, pred, gold_logits, va):
    """Model + predictions. Saved every time the check score improves, so a stopped run keeps the best so far.
    Each file is written under a temporary name first, so a stop in the middle never leaves a broken file."""
    torch.save(state, f"/kaggle/working/v5_fold{fold}.pt.tmp")
    os.replace(f"/kaggle/working/v5_fold{fold}.pt.tmp", f"/kaggle/working/v5_fold{fold}.pt")
    out = pd.DataFrame(pred, columns=LABELS)
    out.insert(0, "StudyInstanceUID", uids_tr[va])
    for i, label in enumerate(LABELS):
        out["t_" + label] = y[va][:, i]
        out["w_" + label] = weight[va][:, i]
    out.to_csv(f"/kaggle/working/oof_fold{fold}.csv.tmp", index=False)
    os.replace(f"/kaggle/working/oof_fold{fold}.csv.tmp", f"/kaggle/working/oof_fold{fold}.csv")
    g = pd.DataFrame(gold_logits, columns=LABELS)
    g.insert(0, "StudyInstanceUID", gold["StudyInstanceUID"].values)
    g.to_csv(f"/kaggle/working/gold_fold{fold}.csv.tmp", index=False)
    os.replace(f"/kaggle/working/gold_fold{fold}.csv.tmp", f"/kaggle/working/gold_fold{fold}.csv")


def hours_used():
    return (time.time() - RUN_START) / 3600


def train_fold(fold):
    torch.manual_seed(SEED + fold)
    tr, va = np.where(fold_of != fold)[0], np.where(fold_of == fold)[0]
    train_dl = loader(rows_tr[tr], y[tr], weight[tr], True)
    valid_dl = loader(rows_tr[va], y[va], weight[va], False)
    assert len(train_dl) > 0, "fewer training studies than BATCH_SIZE"
    assert len(va) > 0, f"fold {fold} has no check studies"
    log(f"fold {fold}: {len(tr)} training studies = {len(train_dl)} steps per round, {len(va)} check studies")

    set_stage(f"fold {fold}: building {BACKBONE} (the first time it downloads the pretrained weights)")
    try:
        base = KneeNetV5(BACKBONE, pretrained=PRETRAINED, enc_size=ENC_SIZE, drop_path=DROP_PATH, head=HEAD).to(DEVICE)
    except Exception as e:
        raise RuntimeError(f"could not build {BACKBONE} with pretrained weights ({type(e).__name__}: {e}). "
                           "Turn Internet ON (Settings -> Internet), or attach the weights dataset.") from e
    log(f"model ready: {sum(p.numel() for p in base.parameters()) / 1e6:.1f} million weights | {gpu_status()}")
    checkpoint_on = GRAD_CHECKPOINT is True
    base.encoder.set_grad_checkpointing(checkpoint_on)
    log(f"memory-saving checkpointing: {'on' if checkpoint_on else 'off'} | "
        f"GPUs used: {f'{N_GPUS} (one helper thread kept for the whole run)' if N_GPUS > 1 and USE_BOTH_GPUS else 1}")
    net = SteadyDataParallel(base) if N_GPUS > 1 and USE_BOTH_GPUS else base
    ema = copy.deepcopy(base).eval()
    for p in ema.parameters():
        p.requires_grad_(False)

    head = [p for n, p in base.named_parameters() if not n.startswith("encoder.")]
    if LAYER_DECAY:
        # the standard way to fine-tune transformers: the deeper the layer, the faster it learns
        from timm.optim._param_groups import param_groups_layer_decay
        enc_groups = [{"params": g["params"], "lr": LR_BACKBONE * g.get("lr_scale", 1.0), "weight_decay": g["weight_decay"]}
                      for g in param_groups_layer_decay(base.encoder, weight_decay=WEIGHT_DECAY, layer_decay=LAYER_DECAY)]
    else:
        enc_groups = [{"params": list(base.encoder.parameters()), "lr": LR_BACKBONE}]
    opt = torch.optim.AdamW(enc_groups + [{"params": head, "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
    steps, warm = EPOCHS * len(train_dl), len(train_dl)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(steps - warm, 1))))
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    best, best_state, step, bad_steps, round_hours = float("nan"), None, 0, 0, 0.0
    top = []                                   # the best rounds, for the average at the end
    for epoch in range(EPOCHS):
        if hours_used() + round_hours > TIME_LIMIT_HOURS:
            log(f"time limit {TIME_LIMIT_HOURS} h: no time for round {epoch} of fold {fold}; its best model is saved")
            STATE["time_stop"] = True
            break
        net.train()
        set_stage(f"fold {fold} round {epoch}: starting data workers, waiting for the first batch")
        t0, running, done, t_wait, t_last = time.time(), 0.0, 0, 0.0, time.time()
        try:
            for i, (x, plane, pos, valid, target, wt) in enumerate(train_dl):
                t_wait += time.time() - t_last
                if i == 0:
                    log(f"  first batch ready after {time.time() - t0:.1f} s")
                STATE["stage"] = f"fold {fold} round {epoch}: step {i + 1}/{len(train_dl)}"
                try:
                    x, plane, pos, valid = x.to(DEVICE), plane.to(DEVICE), pos.to(DEVICE), valid.to(DEVICE)
                    target, wt = target.to(DEVICE), wt.to(DEVICE)
                    with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                        logits = net(x, plane, pos, valid)
                    loss = (nn.functional.binary_cross_entropy_with_logits(logits.float(), target, reduction="none") * wt).sum() / wt.sum().clamp(min=1)
                    if not torch.isfinite(loss):
                        raise FloatingPointError(f"loss is {loss.item()}")
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(base.parameters(), 2.0)
                    scaler.step(opt)
                    scaler.update()
                except (torch.cuda.OutOfMemoryError, FloatingPointError) as e:
                    # skip this batch, but stop if it keeps happening
                    if isinstance(e, torch.cuda.OutOfMemoryError) and GRAD_CHECKPOINT == "auto" and not checkpoint_on:
                        checkpoint_on = True                   # GPU too small without it: switch it on and go on
                        base.encoder.set_grad_checkpointing(True)
                        log("  GPU out of memory: memory-saving checkpointing switched on (slower, but it fits)")
                        bad_steps -= 1                         # this one does not count as a bad step
                    bad_steps += 1
                    logits = loss = None                       # drop the half-built graph so its memory can be freed
                    opt.zero_grad(set_to_none=True)
                    scaler = torch.amp.GradScaler("cuda", init_scale=scaler.get_scale(), enabled=(DEVICE == "cuda"))
                    if DEVICE == "cuda":
                        torch.cuda.empty_cache()
                    log(f"  step {i + 1} skipped ({type(e).__name__}: {e}); skipped so far: {bad_steps}")
                    if bad_steps > MAX_BAD_STEPS:
                        raise RuntimeError("too many skipped steps: lower BATCH_SIZE or TRAIN_PICTURES") from e
                    t_last = time.time()
                    continue
                opt.zero_grad(set_to_none=True)
                sched.step()
                step += 1
                d = min(EMA_DECAY, (1 + step) / (10 + step))
                with torch.no_grad():
                    for e_, m_ in zip(ema.state_dict().values(), base.state_dict().values()):
                        if e_.dtype.is_floating_point:
                            e_.mul_(d).add_(m_.detach(), alpha=1 - d)
                        else:
                            e_.copy_(m_)
                running += loss.item()                     # .item() waits for the GPU, so the timing below is real
                done += 1
                STATE["last_progress"] = t_last = time.time()
                if i == 0:
                    log(f"  first step done after {time.time() - t0:.1f} s | {gpu_status()}")
                if (i + 1) % PRINT_EVERY == 0:
                    free_memory()                              # memory grew on every step on Kaggle, so clean up often
                    if memory_too_high():
                        STATE["memory_stop"] = True
                        log(f"  memory too high in the middle of a round ({memory_text()}): stopping this fold; "
                            "models saved in earlier rounds are kept")
                        break
                    spent = time.time() - t0
                    left = spent / (i + 1) * (len(train_dl) - i - 1) / 60
                    log(f"  step {i + 1}/{len(train_dl)}  loss {running / done:.4f}  {spent / (i + 1):.2f} s/step  "
                        f"waiting for data {t_wait / spent:.0%}  about {left:.0f} min left | {memory_text()}")
        except RuntimeError as e:
            if "timed out" in str(e):
                raise RuntimeError(f"no data batch for {LOADER_TIMEOUT} s: data loading is stuck. "
                                   "Try NUM_WORKERS = 0 to see the real error.") from e
            raise
        if STATE.get("memory_stop"):
            break
        if done == 0:
            raise RuntimeError(f"fold {fold} round {epoch}: every step failed")

        set_stage(f"fold {fold} round {epoch}: scoring {len(va)} check studies")
        pred = sigmoid(predict(ema, valid_dl, "check studies"))
        score = macro_auc(y[va], pred, (weight[va] > 0.5).astype("float32"))["MACRO"]
        set_stage(f"fold {fold} round {epoch}: scoring the {len(y_gold)} expert studies")
        gold_logits = predict(ema, gold_dl, "expert studies")
        g = macro_auc(y_gold, sigmoid(gold_logits))["MACRO"]
        mins = (time.time() - t0) / 60
        round_hours = mins / 60
        log(f"fold {fold} round {epoch:2d}  loss {running / done:.4f}  check {score:.3f}  expert {g:.3f}  {mins:.1f} min  "
            f"waiting for data {t_wait / max(mins * 60, 1e-9):.0%}")
        if epoch == 0:
            log(f"  estimate: about {mins * EPOCHS / 60:.1f} h for this fold ({EPOCHS} rounds)")

        if SWA_TOP > 1:
            top.append((score, {k: v.detach().half().cpu().clone() if v.dtype.is_floating_point else v.cpu().clone()
                                for k, v in ema.state_dict().items()}))
            top = sorted(top, key=lambda t: -t[0] if np.isfinite(t[0]) else 1e9)[:SWA_TOP]
        if best_state is None or np.isnan(best) or score > best:     # the first round is always kept
            best = score
            best_state = {k: v.detach().half().cpu().clone() if v.dtype.is_floating_point else v.cpu().clone()
                          for k, v in ema.state_dict().items()}
            save_fold(fold, best_state, pred, gold_logits, va)
            log(f"  saved: best check score so far {best:.3f}")
        free_memory()
        log(f"  memory after cleanup: {memory_text()}")
        if memory_too_high():
            STATE["memory_stop"] = True
            log(f"memory above MEM_STOP_GB = {MEM_STOP_GB}: stopping training cleanly; the best models are saved")
            break
        if hours_used() > TIME_LIMIT_HOURS:
            log(f"time limit {TIME_LIMIT_HOURS} h reached: fold {fold} stops after round {epoch}; its best model is saved")
            break

    if SWA_TOP > 1 and len(top) >= 2:
        # average the weights of the best rounds: costs nothing at prediction time, usually a little better
        avg = {k: (sum(t[1][k].float() for t in top) / len(top)).half() if v.dtype.is_floating_point else v.clone()
               for k, v in top[0][1].items()}
        ema.load_state_dict(avg)
        set_stage(f"fold {fold}: scoring the average of the best {len(top)} rounds")
        pred = sigmoid(predict(ema, valid_dl, "check studies (averaged model)"))
        score = macro_auc(y[va], pred, (weight[va] > 0.5).astype("float32"))["MACRO"]
        gold_logits = predict(ema, gold_dl, "expert studies (averaged model)")
        g = macro_auc(y_gold, sigmoid(gold_logits))["MACRO"]
        log(f"fold {fold} averaged model (rounds with check {[round(float(t[0]), 3) for t in top]}): "
            f"check {score:.3f}  expert {g:.3f}   best single round: check {best:.3f}")
        save_fold(fold, avg, pred, gold_logits, va)
        best = score
        log("  saved: the averaged model is this fold's model")
    del top
    del net, base, ema, opt, sched, scaler, train_dl, valid_dl
    free_memory()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return best


json.dump({"backbone": BACKBONE, "family": FAMILY, "prep": prep, "labels": LABELS, "folds_id": FOLDS_ID, "enc_size": ENC_SIZE, "head": HEAD,
           "label_round": 2 if USE_ROUND2 else 1},
          open("/kaggle/working/config.json", "w"), indent=1)

fold_scores = {}
for k, fold in enumerate(FOLDS_TO_RUN):
    used = hours_used()
    if k > 0 and used + used / k > TIME_LIMIT_HOURS:          # one more fold would not fit
        log(f"stopping before fold {fold}: about {used / k:.1f} h per fold and {used:.1f} h used. Run it in a new session.")
        break
    if used > TIME_LIMIT_HOURS or STATE.get("memory_stop") or STATE.get("time_stop"):
        break
    fold_scores[fold] = train_fold(fold)
    log(f"--- fold {fold} done: best check score {fold_scores[fold]:.3f}   {hours_used():.2f} h so far | {memory_text()}")
set_stage(f"training finished, folds done in this session: {list(fold_scores)}")

[    0.6 min] fold 0: 3479 training studies = 869 steps per round, 870 check studies
[    0.6 min] fold 0: building tf_efficientnetv2_s.in21k_ft_in1k (the first time it downloads the pretrained weights)


model.safetensors: reconstructing file:   0%|          |  0.00B / 86.5MB            

model.safetensors: downloading bytes:           |  0.00B            

[    0.7 min] model ready: 21.6 million weights | GPU0 3% busy, 0.2/15 GB | GPU1 0% busy, 0.0/15 GB
[    0.7 min] memory-saving checkpointing: off | GPUs used: 2 (one helper thread kept for the whole run)
[    0.7 min] fold 0 round 0: starting data workers, waiting for the first batch
[    0.7 min]   first batch ready after 0.8 s
[    0.9 min]   first step done after 11.9 s | GPU0 10% busy, 5.0/15 GB | GPU1 0% busy, 4.8/15 GB
[    1.2 min]   step 50/869  loss 0.6595  0.64 s/step  waiting for data 5%  about 9 min left | RAM 4.4 GB = programs 2.5 + shared 0.4 + file cache 1.4 | main process 2.9 GB
[    1.5 min]   step 100/869  loss 0.6072  0.52 s/step  waiting for data 5%  about 7 min left | RAM 4.6 GB = programs 2.5 + shared 0.4 + file cache 1.6 | main process 2.9 GB
[    1.9 min]   step 150/869  loss 0.5847  0.48 s/step  waiting for data 5%  about 6 min left | RAM 4.8 GB = programs 2.5 + shared 0.4 + file cache 1.8 | main process 2.9 GB
[    2.0 min] alive | fold 0 round 0: step 171/86

## 6. Results of the folds trained in this session

In [11]:
set_stage("final scores")
oof_files = sorted(glob.glob("/kaggle/working/oof_fold*.csv"))
if not oof_files:
    log("no fold finished in this session, so there is nothing to score")
else:
    oof = pd.concat([pd.read_csv(p) for p in oof_files])
    index = {u: i for i, u in enumerate(uids_tr)}
    oof = oof[oof["StudyInstanceUID"].astype(str).isin(index)]
    ii = np.array([index[u] for u in oof["StudyInstanceUID"].astype(str)])
    oof_scores = macro_auc(y[ii], oof[LABELS].values, (weight[ii] > 0.5).astype("float32"))
    print("OOF score per finding (report labels, studies never seen by their model):")
    show(oof_scores)

    logits = []
    model_files = sorted(glob.glob("/kaggle/working/v5_fold*.pt"))
    for path in model_files:
        m = KneeNetV5(BACKBONE, pretrained=False, enc_size=ENC_SIZE, head=HEAD).to(DEVICE)
        m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in torch.load(path, map_location=DEVICE).items()})
        logits.append(predict(m, gold_dl, f"expert studies with {os.path.basename(path)}"))
        del m
    gold_scores = macro_auc(y_gold, sigmoid(np.mean(logits, axis=0))) if logits else {"MACRO": float("nan")}
    print(f"\ngold score per finding ({len(logits)} fold models averaged):")
    show(gold_scores)
    folds_done = [int(os.path.basename(p)[len("oof_fold"):-4]) for p in oof_files]

    if MIRROR_CHECK and model_files:
        # same models, same studies: normal copy only vs normal + mirrored copy
        log("mirror check: scoring a mirrored copy of the check and expert studies")
        plain_oof, both_oof, idx_all, gold_mirror = [], [], [], []
        for path in model_files:
            k = int(os.path.basename(path)[len("v5_fold"):-3])
            m = KneeNetV5(BACKBONE, pretrained=False, enc_size=ENC_SIZE, head=HEAD).to(DEVICE)
            m.load_state_dict({kk: v.float() if v.is_floating_point() else v for kk, v in torch.load(path, map_location=DEVICE).items()})
            va = np.where(fold_of == k)[0]
            a = predict(m, loader(rows_tr[va], y[va], weight[va], False), f"check studies, part {k}")
            b = predict(m, loader(rows_tr[va], y[va], weight[va], False, mirror=True), f"mirrored check studies, part {k}")
            plain_oof.append(a); both_oof.append(join_mirror(a, b)); idx_all.append(va)
            gold_mirror.append(predict(m, loader(rows_gold, y_gold, np.ones_like(y_gold), False, mirror=True), "mirrored expert studies"))
            del m
        ii = np.concatenate(idx_all)
        mk = (weight[ii] > 0.5).astype("float32")
        s_plain = macro_auc(y[ii], np.concatenate(plain_oof), mk)
        s_both = macro_auc(y[ii], np.concatenate(both_oof), mk)
        g_plain = macro_auc(y_gold, sigmoid(np.mean(logits, axis=0)))
        g_both = macro_auc(y_gold, sigmoid(join_mirror(np.mean(logits, axis=0), np.mean(gold_mirror, axis=0))))
        print("\nmirror check (normal only -> normal + mirrored), per finding:")
        for name in LABELS + ["MACRO"]:
            if name in s_plain and name in s_both:
                print(f"  {name:18s} check {s_plain[name]:.3f} -> {s_both[name]:.3f}   expert "
                      f"{g_plain.get(name, float('nan')):.3f} -> {g_both.get(name, float('nan')):.3f}")
        log(f"MIRROR CHECK  check {s_plain['MACRO']:.4f} -> {s_both['MACRO']:.4f}   "
            f"expert {g_plain['MACRO']:.4f} -> {g_both['MACRO']:.4f}  "
            "(use MIRROR_TTA = True in the prediction notebook only if both go up)")
    log(f"RESULT  v5 {FAMILY} folds {folds_done}  size {prep['img_size']}  "
        f"OOF {oof_scores['MACRO']:.4f}   gold {gold_scores['MACRO']:.4f}")

[  500.2 min] final scores
OOF score per finding (report labels, studies never seen by their model):
  ACL                0.982
  MCL                0.978
  Medial Meniscus    0.980
  Lateral Meniscus   0.972
  Medial OA          0.972
  Lateral OA         0.960
  PF OA              0.955
  Effusion           0.974
  Synovitis          0.964
  Baker's            0.984
  Contusion          0.973
  Fracture           0.964
  MACRO              0.971
[  500.2 min] alive | final scores | last progress 0 min ago | RAM 6.3 GB = programs 2.4 + shared 0.1 + file cache 3.7 | main process 3.1 GB | GPU0 0% busy, 0.4/15 GB | GPU1 0% busy, 0.3/15 GB
[  500.4 min]   scoring expert studies with v5_fold0.pt: batch 15/15  0.54 s per batch
[  500.5 min]   scoring expert studies with v5_fold1.pt: batch 15/15  0.52 s per batch
[  500.7 min]   scoring expert studies with v5_fold2.pt: batch 15/15  0.54 s per batch
[  500.8 min]   scoring expert studies with v5_fold3.pt: batch 15/15  0.53 s per batch
[  501.